# Phase 6 -- Transaction / Rule Model Experiments

Alert Intelligence Engine -- master plan Phase 6 (section 20): "IF/AE/benchmarks + validation." Gate: champion selected.

Second (and final PoC-stage) model-training phase. Trains and compares anomaly/novelty models on the Phase 4 transaction representation (`Rule` sheet -- Transaction Rule Alerts).

**PII safety note** (same discipline as Phase 5, after the Phase 2 incident): this notebook never prints `Customer Name`, `Beneficiary Name`, or `Customer Number`. Inspection tables use `record_id` + non-identifying context only.

**Non-negotiable rules in force:**
- Isolation Forest is the starting candidate, not a pre-declared winner -- benchmarked against Autoencoder, LOF, One-Class SVM.
- No classification accuracy. `Status` (Released/UPS/Followup) appears only in one clearly-labeled exploratory cell at the end, never in selection.
- Group-split (unseen customers) and time-forward split reported separately.
- **Do not train 18 independent rule models** (master plan section 6): one shared transaction representation, Rule Name as context. Per-rule behaviour is *evaluated* (novelty by rule segment), never separately *modeled*.
- Transaction amount stays excluded (Phase 4 finding: no structured field, only in the leakage-typed `Comment`).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import json
import time
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Load Phase 2 normalized Rule data

In [2]:
from pipelines.normalization.pipeline import run_phase2_pipeline

normalized_sheets, phase2_report = run_phase2_pipeline(REPO_ROOT, persist=False)
assert phase2_report["overall_status"] == "PASS"

rule_df = normalized_sheets["Rule"]
print(f"Rule: {len(rule_df)} rows")
print(rule_df["Rule Name"].value_counts())

Rule: 2244 rows
Rule Name
23 - NATIONALITY - COUNTRY COMBINATION (SEND TRANS           640
42 - REMITTANCE - NON RESIDENT                               365
53 - RULE TRANSACTIONS BREACHED AMOUNT 55K                   336
107 - AMOUNT EXCEEDS 30%                                     331
91 - PEP CUSTOMER                                            122
56 - HIGH RISK COUNTRY - CUMULATIVE                          116
84 - TRANSACTIONS BREACHED AMOUNT 55K WITH IN A 90 DAYS      102
4 - HIGH VALUE TRANSACTIONS                                   52
96 - MULTIPLE BENEFICIARIES (ONE TO MANY) SEND                39
44 - MULTIPLE SENDERS (MANY TO ONE)                           34
97 - SPLITTING OF TRANSACTIONS                                33
127 - SEND TO HIGH RISK COUNTRY                               24
20 - HIGH VALUE TRANSACTIONS - CUMULATIVE                     15
100 - MANY CUSTOMERS RECEIVING FUNDS FROM A COMMON SENDER     11
104 - NEW CLIENT HIGH VALUE TRANSACTION                       10

## 2. Two validation scenarios

In [3]:
from pipelines.entity.validation_splits import group_split_by_customer, time_forward_split

group_train_idx, group_test_idx = group_split_by_customer(rule_df, test_size=0.25)
time_train_idx, time_test_idx = time_forward_split(rule_df, "Scan Date (Parsed)", test_frac=0.25)

scenarios = {
    "unseen_customers": (group_train_idx, group_test_idx),
    "repeat_customers_time_forward": (time_train_idx, time_test_idx),
}
for name, (tr, te) in scenarios.items():
    print(f"{name}: train={len(tr)}, test={len(te)}")

unseen_customers: train=1599, test=596
repeat_customers_time_forward: train=1683, test=561


## 3. Fit representations per scenario

T1 uses the structured (categorical + nationality) block only. T2-T5 explicitly share one representation -- the full behavioural feature matrix (context + Beneficiary Name representation + leakage-safe prior-alert/prior-rule-diversity history), SVD-reduced for tractability.

In [4]:
from features.transaction_features import fit_transaction_feature_artifacts, transform_transaction_features
from pipelines.transaction.anomaly_models import extract_structured_matrix
from pipelines.entity.anomaly_models import fit_svd

scenario_data = {}

for scenario_name, (train_idx, test_idx) in scenarios.items():
    train_df = rule_df.iloc[train_idx].reset_index(drop=True)
    test_df = rule_df.iloc[test_idx].reset_index(drop=True)

    artifacts = fit_transaction_feature_artifacts(train_df)
    train_matrix, block_names = transform_transaction_features(train_df, artifacts)
    artifacts.feature_names = block_names
    test_matrix, _ = transform_transaction_features(test_df, artifacts)

    structured_train = extract_structured_matrix(train_matrix, artifacts)
    structured_test = extract_structured_matrix(test_matrix, artifacts)

    svd = fit_svd(train_matrix, n_components=50)
    svd_train = svd.transform(train_matrix)
    svd_test = svd.transform(test_matrix)

    scenario_data[scenario_name] = {
        "train_df": train_df, "test_df": test_df, "artifacts": artifacts,
        "structured_train": structured_train, "structured_test": structured_test,
        "svd_train": svd_train, "svd_test": svd_test, "svd": svd,
    }
    print(f"{scenario_name}: structured dims={structured_train.shape[1]}, "
          f"full-representation SVD dims={svd_train.shape[1]}")

unseen_customers: structured dims=191, full-representation SVD dims=50


repeat_customers_time_forward: structured dims=205, full-representation SVD dims=50


## 4. Experiment matrix -- T1 through T5, both scenarios

In [5]:
from pipelines.entity.anomaly_models import (
    fit_isolation_forest, score_isolation_forest,
    fit_autoencoder, score_autoencoder,
    fit_lof, score_lof,
    fit_ocsvm, score_ocsvm,
    RANDOM_STATE,
)
from pipelines.entity.evaluation import (
    score_distribution_summary, ranking_stability_between_models,
    customer_history_consistency, ExperimentResult,
)

MODEL_DEFS = {
    "T1_isolation_forest_structured": ("structured", "isolation_forest"),
    "T2_isolation_forest_behavioural_svd": ("behavioural_svd", "isolation_forest"),
    "T3_autoencoder_behavioural_svd": ("behavioural_svd", "autoencoder"),
    "T4_lof_behavioural_svd": ("behavioural_svd", "lof"),
    "T5_ocsvm_behavioural_svd": ("behavioural_svd", "ocsvm"),
}

def fit_and_score(model_kind, X_train, X_test):
    t0 = time.time()
    if model_kind == "isolation_forest":
        model = fit_isolation_forest(X_train); fit_s = time.time() - t0
        t1 = time.time(); scores = score_isolation_forest(model, X_test)
    elif model_kind == "autoencoder":
        model = fit_autoencoder(X_train, bottleneck=8, epochs=100); fit_s = time.time() - t0
        t1 = time.time(); scores = score_autoencoder(model, X_test)
    elif model_kind == "lof":
        model = fit_lof(X_train); fit_s = time.time() - t0
        t1 = time.time(); scores = score_lof(model, X_test)
    elif model_kind == "ocsvm":
        model = fit_ocsvm(X_train); fit_s = time.time() - t0
        t1 = time.time(); scores = score_ocsvm(model, X_test)
    else:
        raise ValueError(model_kind)
    return model, scores, fit_s, time.time() - t1

rng = np.random.default_rng(RANDOM_STATE)
results = []
fitted_models = {}

for scenario_name, data in scenario_data.items():
    for exp_id, (repr_name, model_kind) in MODEL_DEFS.items():
        X_train = data["structured_train"] if repr_name == "structured" else data["svd_train"]
        X_test = data["structured_test"] if repr_name == "structured" else data["svd_test"]

        model, scores, fit_s, score_s = fit_and_score(model_kind, X_train, X_test)

        n_train = X_train.shape[0]
        boot_idx = rng.choice(n_train, size=int(n_train * 0.8), replace=False)
        _, scores_boot, _, _ = fit_and_score(model_kind, X_train[boot_idx], X_test)
        stability = ranking_stability_between_models(scores, scores_boot)

        dist = score_distribution_summary(scores)
        hist_consistency = customer_history_consistency(data["test_df"], scores)

        result = ExperimentResult(
            experiment_id=f"{scenario_name}::{exp_id}",
            model_name=model_kind, representation=repr_name, validation_scenario=scenario_name,
            distribution=dist, stability_spearman=stability, history_consistency=hist_consistency,
            fit_seconds=round(fit_s, 3), score_seconds=round(score_s, 3),
        )
        results.append(result)
        fitted_models[result.experiment_id] = (model, scores)
        print(f"{result.experiment_id}: std={dist['std']:.4f}, stability={stability:.3f}, "
              f"fit={fit_s:.2f}s, score={score_s:.3f}s")

unseen_customers::T1_isolation_forest_structured: std=0.0136, stability=0.883, fit=0.31s, score=0.005s


unseen_customers::T2_isolation_forest_behavioural_svd: std=0.0358, stability=0.976, fit=0.35s, score=0.007s


/home/chpl/Documents/AI-validation/AI_validator/Alert-AI/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:619: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


unseen_customers::T3_autoencoder_behavioural_svd: std=0.4691, stability=0.997, fit=4.84s, score=0.000s
unseen_customers::T4_lof_behavioural_svd: std=0.9133, stability=0.883, fit=0.16s, score=0.004s


unseen_customers::T5_ocsvm_behavioural_svd: std=1.8647, stability=0.998, fit=0.01s, score=0.004s


repeat_customers_time_forward::T1_isolation_forest_structured: std=0.0157, stability=0.926, fit=0.36s, score=0.005s


repeat_customers_time_forward::T2_isolation_forest_behavioural_svd: std=0.0364, stability=0.976, fit=0.35s, score=0.007s


repeat_customers_time_forward::T3_autoencoder_behavioural_svd: std=0.4397, stability=0.995, fit=0.18s, score=0.001s
repeat_customers_time_forward::T4_lof_behavioural_svd: std=1.6071, stability=0.919, fit=0.02s, score=0.004s
repeat_customers_time_forward::T5_ocsvm_behavioural_svd: std=0.8166, stability=0.997, fit=0.02s, score=0.004s


## 5. Results table

In [6]:
results_df = pd.DataFrame([r.as_dict() for r in results])
results_df["std"] = results_df["distribution"].apply(lambda d: d["std"])
results_df["is_degenerate"] = results_df["distribution"].apply(lambda d: d["is_degenerate"])
display_cols = ["experiment_id", "model_name", "representation", "validation_scenario",
                 "std", "is_degenerate", "stability_spearman", "fit_seconds", "score_seconds"]
print(results_df[display_cols].to_string(index=False))

                                                     experiment_id       model_name  representation           validation_scenario      std  is_degenerate  stability_spearman  fit_seconds  score_seconds
                  unseen_customers::T1_isolation_forest_structured isolation_forest      structured              unseen_customers 0.013602          False            0.883126        0.308          0.005
             unseen_customers::T2_isolation_forest_behavioural_svd isolation_forest behavioural_svd              unseen_customers 0.035789          False            0.976413        0.350          0.007
                  unseen_customers::T3_autoencoder_behavioural_svd      autoencoder behavioural_svd              unseen_customers 0.469071          False            0.996686        4.837          0.000
                          unseen_customers::T4_lof_behavioural_svd              lof behavioural_svd              unseen_customers 0.913262          False            0.882619        0.156      

## 6. Champion selection

In [7]:
from pipelines.entity.evaluation import select_champion

champion, rubric = select_champion(results)
print(json.dumps(rubric, indent=2, default=str))
print()
print(f"CHAMPION: {champion.experiment_id}")
print(f"  model: {champion.model_name}, representation: {champion.representation}, "
      f"scenario: {champion.validation_scenario}")

{
  "dropped_degenerate_experiment_ids": [],
  "ranked_experiment_ids_best_to_worst": [
    "unseen_customers::T5_ocsvm_behavioural_svd",
    "repeat_customers_time_forward::T5_ocsvm_behavioural_svd",
    "unseen_customers::T3_autoencoder_behavioural_svd",
    "repeat_customers_time_forward::T3_autoencoder_behavioural_svd",
    "unseen_customers::T2_isolation_forest_behavioural_svd",
    "repeat_customers_time_forward::T2_isolation_forest_behavioural_svd",
    "repeat_customers_time_forward::T1_isolation_forest_structured",
    "repeat_customers_time_forward::T4_lof_behavioural_svd",
    "unseen_customers::T1_isolation_forest_structured",
    "unseen_customers::T4_lof_behavioural_svd"
  ],
  "champion_experiment_id": "unseen_customers::T5_ocsvm_behavioural_svd",
  "champion_stability_spearman": 0.9981192307171267,
  "champion_consistency_ratio": 0.49944976888030684,
  "selection_criteria_order": [
    "not degenerate",
    "highest bootstrap ranking stability (Spearman)",
    "lowest w

## 7. Per-rule stability -- master plan requirement specific to Transaction

"Benchmark anomaly methods separately by rule family where sample size permits" and "Prevent dominant rules from hiding weak segments." This is *evaluation*, not 18 separate models -- the champion is one shared model; here we check whether its novelty behaviour is stable across rule types, or whether a dominant rule (e.g. rule 23, ~28% of the sample) is skewing the picture.

In [8]:
from pipelines.entity.evaluation import novelty_by_segment, top_n_novelty_table, score_stability_across_time

champion_scenario = champion.validation_scenario
_, champion_scores = fitted_models[champion.experiment_id]
champion_test_df = scenario_data[champion_scenario]["test_df"]

per_rule = novelty_by_segment(champion_test_df, champion_scores, "Rule Name")
print(per_rule)
print()
small_sample_rules = per_rule[per_rule["count"] < 5]
if len(small_sample_rules):
    print(f"NOTE: {len(small_sample_rules)} rule(s) have <5 test-set alerts -- their mean/std "
          f"novelty is not statistically reliable at this sample size, reported for completeness only:")
    print(small_sample_rules)

                                                         mean       std  count
Rule Name                                                                     
56 - HIGH RISK COUNTRY - CUMULATIVE                -45.405406  4.191916     40
91 - PEP CUSTOMER                                  -46.029364  3.459813     32
101 - MULTIPLE CURRENCIES                          -47.503141  0.332190      2
20 - HIGH VALUE TRANSACTIONS - CUMULATIVE          -47.922383  0.501508      6
42 - REMITTANCE - NON RESIDENT                     -47.942172  0.705241     94
107 - AMOUNT EXCEEDS 30%                           -48.091598  0.772144     87
23 - NATIONALITY - COUNTRY COMBINATION (SEND TRANS -48.146261  0.845772    170
97 - SPLITTING OF TRANSACTIONS                     -48.208873  0.516025      8
84 - TRANSACTIONS BREACHED AMOUNT 55K WITH IN A... -48.263149  0.828233     16
53 - RULE TRANSACTIONS BREACHED AMOUNT 55K         -48.399659  1.612532    112
96 - MULTIPLE BENEFICIARIES (ONE TO MANY) SEND     -

## 8. Manual inspection -- non-PII columns only

In [9]:
SAFE_DISPLAY_COLS = [
    "record_id", "Rule Name", "Transaction Type Code", "Branch Description",
    "Currency Name", "Beneficiary Relationship",
]
top_bottom = top_n_novelty_table(champion_test_df, champion_scores, SAFE_DISPLAY_COLS, n=10)
print(top_bottom.to_string(index=False))
print()
print("=== Score stability across time windows ===")
print(score_stability_across_time(champion_test_df, champion_scores, "Scan Date (Parsed)", n_bins=4))

       record_id                                          Rule Name Transaction Type Code                         Branch Description     Currency Name Beneficiary Relationship  novelty_score  novelty_rank  group
c2beeb3452c620b7                56 - HIGH RISK COUNTRY - CUMULATIVE                 ORMTN                       MAZAYA CENTRE BRANCH         US DOLLAR                   FRIEND     -35.995194           1.0    top
ec1c4ee275b74fe6                56 - HIGH RISK COUNTRY - CUMULATIVE                 ORMTN                       MAZAYA CENTRE BRANCH         US DOLLAR                   FRIEND     -36.702303           2.0    top
709a37d9230a26f1                56 - HIGH RISK COUNTRY - CUMULATIVE                 ORMTN                       MAZAYA CENTRE BRANCH         US DOLLAR                   FRIEND     -37.506891           3.0    top
3667d14d0f68ba8c                                  91 - PEP CUSTOMER                 FRXNB                               DUBAI BRANCH        UAE DIRHAM  

## 9. Exploratory-only comparison against historical Status

**Does not feed champion selection.** `Status` is a raw column on `rule_df` that was never included in any feature list (Phase 4's leakage guard) -- referenced here directly (no join needed; `champion_test_df` is a positional slice of `rule_df`, so the column is already present and unambiguous, unlike Phase 5's combined-dataset case which needed the record_id-collision workaround).

In [10]:
from pipelines.entity.evaluation import exploratory_status_comparison

print(exploratory_status_comparison(champion_test_df, champion_scores, "Status"))
print()
print("Reminder: Status is not a confirmed true-match label -- see Phase 1 label audit.")
print("This table is context, not evidence the model 'works' or 'doesn't work'.")

               mean       std  count
Status                              
Released -47.794024  1.910268    523
UPS      -48.714220  1.254212     73

Reminder: Status is not a confirmed true-match label -- see Phase 1 label audit.
This table is context, not evidence the model 'works' or 'doesn't work'.


## 10. Persist champion model

In [11]:
import joblib

champion_model, _ = fitted_models[champion.experiment_id]
out_dir = REPO_ROOT / "models" / "transaction"
out_dir.mkdir(parents=True, exist_ok=True)

champion_path = out_dir / f"champion_{champion.experiment_id.replace('::', '_')}.joblib"
payload = {
    "model": champion_model,
    "model_kind": champion.model_name,
    "representation": champion.representation,
    "svd": scenario_data[champion_scenario]["svd"] if champion.representation == "behavioural_svd" else None,
    "transaction_feature_artifacts": scenario_data[champion_scenario]["artifacts"],
    "validation_scenario": champion_scenario,
    "experiment_id": champion.experiment_id,
}
joblib.dump(payload, champion_path)
print(f"Champion persisted (gitignored, local-only): {champion_path}")

champion_manifest = {
    "experiment_id": champion.experiment_id,
    "model_kind": champion.model_name,
    "representation": champion.representation,
    "validation_scenario": champion_scenario,
    "selection_rubric": rubric,
    "distribution": champion.distribution,
    "history_consistency": champion.history_consistency,
    "per_rule_novelty": per_rule.to_dict(orient="index"),
    "artifact_file": champion_path.name,
}
manifest_path = out_dir / "champion_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(champion_manifest, f, indent=2, default=str)
print(f"Champion manifest (gitignored): {manifest_path}")

Champion persisted (gitignored, local-only): /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/models/transaction/champion_unseen_customers_T5_ocsvm_behavioural_svd.joblib
Champion manifest (gitignored): /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/models/transaction/champion_manifest.json


## 11. Phase 6 report (aggregate only -- no PII)

In [12]:
phase6_report = {
    "phase": "6_transaction_experiments",
    "status": "PASS",
    "experiments_run": [r.as_dict() for r in results],
    "champion": {
        "experiment_id": champion.experiment_id,
        "model_kind": champion.model_name,
        "representation": champion.representation,
        "validation_scenario": champion_scenario,
    },
    "selection_rubric": rubric,
    "per_rule_novelty_summary": per_rule.to_dict(orient="index"),
    "checks": {
        "isolation_forest_not_assumed_winner": True,
        "autoencoder_benchmarked": True,
        "lof_and_ocsvm_benchmarked": True,
        "both_validation_scenarios_reported_separately": True,
        "no_18_independent_rule_models_trained": True,
        "per_rule_novelty_evaluated_not_modeled_separately": True,
        "no_classification_accuracy_reported": True,
        "status_used_only_in_labeled_exploratory_cell": True,
        "transaction_amount_excluded_no_structured_field": True,
        "no_pii_in_this_report_or_notebook_output": True,
    },
    "known_limitations": [
        "No structured transaction amount -- champion cannot use amount-based behavioural "
        "signal (z-score vs. customer history, etc.) until the client supplies a structured field.",
        "Several rules have small test-set sample sizes (<5 alerts); their per-rule novelty "
        "mean/std is directional only, not statistically reliable at this volume.",
        "T2-T5 share one SVD-reduced representation of the full feature matrix -- champion "
        "evidence is about this reduced representation, not the raw feature space directly.",
    ],
    "next_gate": "Phase 7 -- Historical replay and validation report consolidating Phase 5 + "
        "Phase 6 evidence. Requires human approval.",
}

out_path = REPO_ROOT / "evaluation" / "phase6_transaction_experiments_report.json"
with open(out_path, "w") as f:
    json.dump(phase6_report, f, indent=2, default=str)
print(f"Report written to {out_path}")

Report written to /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/evaluation/phase6_transaction_experiments_report.json


## 12. Test suite

In [13]:
import subprocess

result = subprocess.run(["python", "-m", "pytest", "tests/", "-q"], cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-2000:])
assert result.returncode == 0, "Test suite must pass before Phase 6 is considered done" 

........................................................................ [ 61%]
..............................................                           [100%]
=============================== warnings summary ===============================
tests/test_anomaly_models.py::test_autoencoder_fit_score_shapes
  /home/chpl/Documents/AI-validation/AI_validator/Alert-AI/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:619: UserWarning: Can't initialize NVML
    warnings.warn("Can't initialize NVML")

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
118 passed, 1 warning in 12.02s



## Phase 6 -- Result

**Status: PASS**

- Trained and compared 5 candidates (Isolation Forest on structured features, Isolation Forest / Autoencoder / LOF / One-Class SVM on the shared behavioural representation) under 2 validation scenarios -- 10 experiments total.
- Champion selected via the same documented, accuracy-free rubric as Phase 5.
- Per-rule novelty evaluated (not modeled separately) -- master plan explicitly forbids training 18 independent rule models; Rule Name stays a context feature in one shared model.
- Small-sample rules flagged, not hidden -- their per-rule stats are reported with an explicit reliability caveat.
- Status comparison remains exploratory-only, structurally excluded from selection.
- No PII in any notebook output (Customer Name / Beneficiary Name / Customer Number never printed).
- Champion model + manifest persisted (gitignored, local-only).
- 118+/118+ tests passing.

**Next gate:** Phase 7 -- Historical replay and validation report, consolidating Phase 5 (Entity) and Phase 6 (Transaction) champion evidence into one reproducible evaluation package. Awaiting human approval to proceed.